# Disclaimer
This notebook helps to export the used input variables, base rasters and labels used for modeling different CMA's with different configurations.<br>
The functions can keep the underlying folder structure and export only those files which have not already been exported.<p>

It works as following:
1. Collect files and create a file list without duplicates
2. Export the files to a new folder structure without overwriting existing files

The export works in that way, that all input files have to contain a same folder structure, idially in the src/data folder.<br>
The export will keep the subfolders and export only the files which are not already existing in the target folder.


# Export data

**Packages**

In [8]:
from beak.utilities.io import copy_files
from pathlib import Path 

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

**User** input

In [9]:
# Path to the data
BASE_PATH = files("beak.data")

EXPORT_PATH = "S:/Projekte/Temp/D4_code_release/beak-ta3/src/beak/data"
MODEL_PATH_LIST = [
                   "S:/Projekte/20230082_DARPA_CriticalMAAS_TA3/Bearbeitung/GitHub/beak-ta3/experiments/mag_nickel_nat/",
                   "S:/Projekte/20230082_DARPA_CriticalMAAS_TA3/Bearbeitung/GitHub/beak-ta3/experiments/mvt_nat/",
                   "S:/Projekte/20230082_DARPA_CriticalMAAS_TA3/Bearbeitung/GitHub/beak-ta3/experiments/tunksten_skarn_nat/"
                   ]


In [10]:
# File pattern for saved input data (.txt files)
input_variables = "input_file_list"
input_base_rasters = "base_raster_file_list"
input_labels = "label_file_list"

PATTERN_LIST = [input_variables, input_base_rasters, input_labels]

**Search** for respective files in model folders

In [11]:
from pathlib import Path
def filter_files(model_path_list, pattern_list):
  filtered_file_list = []
  pattern_not_found_list = []
  
  for folder in model_path_list:
    folder = Path(folder)
    file_list = list(folder.rglob("*.txt"))

    for file in file_list:
      if any(pattern in str(file) for pattern in pattern_list):
        filtered_file_list.append(file)
      else:
        pattern_not_found_list.append(file)  

  return filtered_file_list, pattern_not_found_list

filtered_files, pattern_not_found = filter_files(MODEL_PATH_LIST, PATTERN_LIST)
print(f"Found files: {len(filtered_files)}, files without matching pattern: {len(pattern_not_found)}")

if len(pattern_not_found) > 0:
  print("\nFiles without matching pattern:")
  for file in pattern_not_found:
    print(file)

Found files: 31, files without matching pattern: 4

Files without matching pattern:
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\experiments\mag_nickel_nat\som\baseline_final\models\SOM_F14_X50_Y50_CMAX50\exports\cluster_hit_count.txt
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\experiments\mag_nickel_nat\som\baseline_final\models\SOM_F14_X50_Y50_CMAX50\exports\result_geo.txt
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\experiments\mag_nickel_nat\som\baseline_final\models\SOM_F14_X50_Y50_CMAX50\exports\result_som.txt
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\experiments\mag_nickel_nat\som\baseline_final\models\SOM_F14_X50_Y50_CMAX50\exports\RunStats.txt


**Read** these files and create a list of unique files from all models

In [12]:
files_to_copy = []

for file in filtered_files:
  with open(file, "r") as f:
    file_rows = f.readlines()
    
    for row in file_rows:
      row = row.strip()
      if row not in files_to_copy:
        files_to_copy.append(row)

files_to_copy = sorted(files_to_copy)
print(f"Files to copy: {len(files_to_copy)}")

Files to copy: 163


**Copy** files to new base folder

In [13]:
copy_files(files_to_copy, old_base_path=BASE_PATH, new_base_path=EXPORT_PATH)